### visualize cumulative recall curve with transfomers 

In [1]:
# ==============================
# Notebook: Recall@k Curves for N-gram Baselines
# ==============================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
from typing import Dict, List, Tuple

# ------------------------------
# 0) Config
# ------------------------------
DATA_PATH = "processed_data/clusters/screen_clusters_k40.tsv"  # <-- change this
K_LIST = [1, 3, 5, 10,20]
ALPHA_SMOOTH = 1.0
LAMBDA_BACKOFF = 0.2

# If your clusters in the TSV are not 0..K-1 contiguous, we will remap them.
# ------------------------------
# 1) Load traces from TSV
# ------------------------------
def load_traces_tsv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    # Expected columns:
    # screen_key, app_id, trace_id, screen_id, cluster_id
    needed = {"app_id", "trace_id", "cluster_id"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in TSV: {missing}. Found: {list(df.columns)}")
    return df

df = load_traces_tsv(DATA_PATH)
df.head()

# ------------------------------
# 2) Build traces per (app_id, trace_id)
# ------------------------------
def build_traces(df: pd.DataFrame, collapse_consecutive_duplicates: bool = True):
    """
    Returns list of tuples: (app_id, trace_id, seq_of_cluster_ids)
    """
    traces = []
    grouped = df.groupby(["app_id", "trace_id"], sort=False)

    for (app_id, trace_id), g in grouped:
        # Preserve original order as in file; if you have a time/order column, sort by it here
        seq = g["cluster_id"].tolist()

        if collapse_consecutive_duplicates and len(seq) > 1:
            collapsed = [seq[0]]
            for x in seq[1:]:
                if x != collapsed[-1]:
                    collapsed.append(x)
            seq = collapsed

        if len(seq) >= 2:
            traces.append((app_id, trace_id, seq))

    return traces

traces_raw = build_traces(df, collapse_consecutive_duplicates=True)
print("Num traces (len>=2):", len(traces_raw))

# ------------------------------
# 3) Remap cluster IDs to 0..K-1 contiguous
# ------------------------------
def remap_clusters(traces):
    all_ids = []
    for _, _, seq in traces:
        all_ids.extend(seq)
    uniq = sorted(set(all_ids))
    id2new = {old:i for i, old in enumerate(uniq)}
    new2old = {i:old for old, i in id2new.items()}

    remapped = []
    for app_id, trace_id, seq in traces:
        remapped.append((app_id, trace_id, [id2new[x] for x in seq]))

    return remapped, id2new, new2old, len(uniq)

traces, id2new, new2old, K = remap_clusters(traces_raw)
print("K (unique clusters observed):", K)

# ------------------------------
# 4) App-split train/test
# ------------------------------
def app_split(traces, test_frac=0.2, seed=42):
    rng = np.random.default_rng(seed)
    apps = sorted(set(a for a, _, _ in traces))
    rng.shuffle(apps)
    n_test = int(round(len(apps) * test_frac))
    test_apps = set(apps[:n_test])

    train = [t for t in traces if t[0] not in test_apps]
    test  = [t for t in traces if t[0] in test_apps]
    return train, test, test_apps

train_traces, test_traces, test_apps = app_split(traces, test_frac=0.2, seed=42)
print("Train traces:", len(train_traces))
print("Test traces :", len(test_traces))
print("Test apps   :", len(test_apps))

# ------------------------------
# 5) Training: Unigram / Bigram / Trigram (counts)
# ------------------------------
def train_unigram_next(train_traces, K: int, alpha: float = 1.0):
    counts = np.zeros(K, dtype=np.float64)
    for _, _, seq in train_traces:
        for b in seq[1:]:
            counts[b] += 1.0
    counts += alpha
    probs = counts / counts.sum()
    return probs  # shape [K]

def train_bigram_counts(train_traces, K: int, alpha: float = 1.0):
    # counts[a][b] transitions
    counts = np.zeros((K, K), dtype=np.float64)
    for _, _, seq in train_traces:
        for a, b in zip(seq[:-1], seq[1:]):
            counts[a, b] += 1.0
    counts += alpha
    # normalize rows
    probs = counts / counts.sum(axis=1, keepdims=True)
    return probs  # shape [K,K]

def train_trigram_counts(train_traces, K: int, alpha: float = 1.0):
    # counts[(a,b)][c]
    counts = defaultdict(lambda: np.zeros(K, dtype=np.float64))
    for _, _, seq in train_traces:
        if len(seq) < 3:
            continue
        for a, b, c in zip(seq[:-2], seq[1:-1], seq[2:]):
            counts[(a, b)][c] += 1.0

    # apply smoothing to each context that exists
    trigram_probs = {}
    for ctx, vec in counts.items():
        vec = vec + alpha
        trigram_probs[ctx] = vec / vec.sum()
    return trigram_probs  # dict[(a,b)] -> np.array[K]

unigram_probs = train_unigram_next(train_traces, K, alpha=ALPHA_SMOOTH)
bigram_probs  = train_bigram_counts(train_traces, K, alpha=ALPHA_SMOOTH)
trigram_probs = train_trigram_counts(train_traces, K, alpha=ALPHA_SMOOTH)

print("Trigram contexts:", len(trigram_probs))

# ------------------------------
# 6) Backoff scoring for trigram: P(c | a,b) with lambda
#    If trigram context missing -> fallback to bigram row.
#    Use bigram + unigram mixture.
# ------------------------------
def score_next_probs(history_a, history_b, trigram_probs, bigram_probs, unigram_probs,
                     lambda_backoff=0.2):
    """
    Returns probability vector over next cluster, shape [K]
    """
    if (history_a, history_b) in trigram_probs:
        p_tri = trigram_probs[(history_a, history_b)]
        # mix with bigram as extra robustness (optional)
        p_bi = bigram_probs[history_b]
        p = (1 - lambda_backoff) * p_tri + lambda_backoff * p_bi
        return p
    else:
        # no trigram context => bigram + unigram backoff
        p_bi = bigram_probs[history_b]
        p = (1 - lambda_backoff) * p_bi + lambda_backoff * unigram_probs
        return p

# ------------------------------
# 7) Eval: Recall@k (trigram-subset positions only)
# ------------------------------
def eval_recall_at_k_unigram(test_traces, unigram_probs, k_list=(1,3,5,10), trigram_subset=True):
    ranked = np.argsort(-unigram_probs)
    hits = {k: 0 for k in k_list}
    total = 0

    for _, _, seq in test_traces:
        # trigram_subset means only evaluate t>=3 positions (need 2-history),
        # but unigram does not use history; we still align evaluation positions
        start = 2 if trigram_subset else 1
        for t in range(start, len(seq)):
            true_next = seq[t]
            total += 1
            for k in k_list:
                if true_next in ranked[:k]:
                    hits[k] += 1

    recall = {k: (hits[k] / total if total > 0 else 0.0) for k in k_list}
    return recall, total

def eval_recall_at_k_bigram(test_traces, bigram_probs, k_list=(1,3,5,10), trigram_subset=True):
    hits = {k: 0 for k in k_list}
    total = 0

    start = 2 if trigram_subset else 1
    for _, _, seq in test_traces:
        for t in range(start, len(seq)):
            prev = seq[t-1]
            true_next = seq[t]
            probs = bigram_probs[prev]
            ranked = np.argsort(-probs)
            total += 1
            for k in k_list:
                if true_next in ranked[:k]:
                    hits[k] += 1

    recall = {k: (hits[k] / total if total > 0 else 0.0) for k in k_list}
    return recall, total

def eval_recall_at_k_trigram(test_traces, trigram_probs, bigram_probs, unigram_probs,
                            lambda_backoff=0.2, k_list=(1,3,5,10)):
    hits = {k: 0 for k in k_list}
    total = 0

    for _, _, seq in test_traces:
        if len(seq) < 3:
            continue
        for t in range(2, len(seq)):
            a = seq[t-2]
            b = seq[t-1]
            true_next = seq[t]
            probs = score_next_probs(a, b, trigram_probs, bigram_probs, unigram_probs,
                                     lambda_backoff=lambda_backoff)
            ranked = np.argsort(-probs)
            total += 1
            for k in k_list:
                if true_next in ranked[:k]:
                    hits[k] += 1

    recall = {k: (hits[k] / total if total > 0 else 0.0) for k in k_list}
    return recall, total




Num traces (len>=2): 8005
K (unique clusters observed): 40
Train traces: 6397
Test traces : 1608
Test apps   : 1489
Trigram contexts: 1332


In [2]:

import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Dict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# -----------------------------
# Model: small causal Transformer
# -----------------------------
class CausalTransformer(nn.Module):
    def __init__(self, vocab_size: int, max_ctx: int, d_model=256, nhead=4, num_layers=4, dropout=0.1, pad_id=0):
        super().__init__()
        self.max_ctx = max_ctx
        self.pad_id = pad_id

        self.tok = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos = nn.Embedding(max_ctx, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout, batch_first=True
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x, attn_mask):
        # x: (B,T), attn_mask: (B,T) True for real tokens
        B, T = x.shape
        pos_ids = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)

        h = self.tok(x) + self.pos(pos_ids)

        # zero padded positions (stability on some CUDA paths)
        h = h.masked_fill(~attn_mask.unsqueeze(-1), 0.0)

        # boolean causal mask: True means blocked
        causal = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)

        # padding mask: True means ignore key
        key_pad = ~attn_mask

        h = self.enc(h, mask=causal, src_key_padding_mask=key_pad)

        # re-zero padded outputs
        h = h.masked_fill(~attn_mask.unsqueeze(-1), 0.0)

        lengths = attn_mask.long().sum(dim=1)
        if (lengths <= 0).any():
            raise RuntimeError("Empty (all-pad) sequence encountered.")
        last_idx = lengths - 1

        last_h = h[torch.arange(B, device=x.device), last_idx]
        logits = self.head(last_h)
        return logits



In [3]:
import torch
import numpy as np

@torch.no_grad()
def eval_recall_at_k_transformer(
    test_traces,
    checkpoint_path,
    device="cuda",
    k_list=(1, 3, 5, 10),
    trigram_subset=True,
):
    """
    Transformer evaluation that matches the trigram/bigram evaluators style.

    Args:
        test_traces: list of (app_id, trace_id, seq_of_cluster_ids)
        checkpoint_path: path to .pt checkpoint saved by transformer training
        device: "cuda" or "cpu"
        k_list: tuple of k values (1,3,5,10)
        trigram_subset:
            - True: evaluate only positions where history exists (t>=2),
                    i.e., matched-context subset used by trigram.
            - False: evaluate on all bigram edges (t>=1).

    Returns:
        recall: dict {k: recall@k}
        total: number of evaluated edges
    """
    ckpt = torch.load(checkpoint_path, map_location="cpu")

    # --- read checkpoint metadata ---
    K = int(ckpt["K"])
    pad_id = int(ckpt["pad_id"])
    bos_id = int(ckpt["bos_id"])
    cfg = ckpt.get("cfg", {})
    max_ctx = int(cfg.get("max_ctx", 16))

    # model hyperparams: must match training
    # if you saved these in cfg, use them; otherwise set defaults used in training
    d_model = int(cfg.get("d_model", 256))
    nhead = int(cfg.get("nhead", 4))
    num_layers = int(cfg.get("num_layers", 4))
    dropout = float(cfg.get("dropout", 0.1))

    vocab = K + 2

    # --- rebuild model + load weights ---
    model = CausalTransformer(
        vocab_size=vocab,
        max_ctx=max_ctx,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dropout=dropout,
        pad_id=pad_id,
    ).to(device)

    model.load_state_dict(ckpt["state_dict"], strict=True)
    model.eval()

    hits = {k: 0 for k in k_list}
    total = 0

    # if trigram_subset=True, require TWO REAL previous clusters.
    # - with BOS at index 0, the first position that has two real predecessors is t=3.
    start_t = 3 if trigram_subset else 1

    for _, _, seq in test_traces:

         # original seq is only real clusters
        if trigram_subset and len(seq) < 3:
            continue

        # prepend BOS (same training setup)
        seq = [bos_id] + list(seq)


        for t in range(start_t, len(seq)):
            prefix = seq[:t]
            true_next = seq[t]

            # build single-item batch with RIGHT padding
            T = min(max_ctx, len(prefix))
            x = torch.full((1, T), pad_id, dtype=torch.long, device=device)
            attn = torch.zeros((1, T), dtype=torch.bool, device=device)

            p = prefix[-T:]
            L = len(p)
            # RIGHT padding
            x[0, :L] = torch.tensor(p, dtype=torch.long, device=device)
            attn[0, :L] = True

            logits = model(x, attn)          # (1, vocab)
            logits = logits[0, :K]           # only real clusters [0..K-1]
            ranked = torch.argsort(logits, descending=True).detach().cpu().numpy()

            total += 1
            for k in k_list:
                if true_next in ranked[:k]:
                    hits[k] += 1

    recall = {k: (hits[k] / total if total > 0 else 0.0) for k in k_list}
    return recall, total


In [ ]:
  

rec_u, tot_u = eval_recall_at_k_unigram(test_traces, unigram_probs, K_LIST, trigram_subset=True)
rec_b, tot_b = eval_recall_at_k_bigram(test_traces, bigram_probs, K_LIST, trigram_subset=True)
rec_t, tot_t = eval_recall_at_k_trigram(test_traces, trigram_probs, bigram_probs, unigram_probs,
                                        lambda_backoff=LAMBDA_BACKOFF, k_list=K_LIST)

print("Edges evaluated (unigram):", tot_u)
print("Edges evaluated (bigram) :", tot_b)
print("Edges evaluated (trigram):", tot_t)

print("Unigram Recall@k:", rec_u)
print("Bigram  Recall@k:", rec_b)
print("Trigram Recall@k:", rec_t)

Edges evaluated (unigram): 6058
Edges evaluated (bigram) : 6058
Edges evaluated (trigram): 6058
Unigram Recall@k: {1: 0.060911191812479365, 3: 0.15912842522284582, 5: 0.24694618686034994, 10: 0.4517992736876857, 20: 0.7256520303730604}
Bigram  Recall@k: {1: 0.12198745460548036, 3: 0.2692307692307692, 5: 0.38478045559590623, 10: 0.5747771541762958, 20: 0.8172664245625619}
Trigram Recall@k: {1: 0.21492241663915485, 3: 0.3626609442060086, 5: 0.463519313304721, 10: 0.6310663585341697, 20: 0.8367448002641136}


In [8]:
rec_tf, tot_tf = eval_recall_at_k_transformer(
    test_traces,
    #checkpoint_path="experiments_weights/best_val_0.7187_transformer_next_cluster_k40_16_200_0.2_8_0.3_256_3e-05_0.1_0.0.pt",
    #checkpoint_path="experiments_weights/best_val_0.8236_transformer_next_cluster_k20_16_200_0.2_8_0.3_256_3e-05_0.1_0.0.pt",
    checkpoint_path="experiments_weights/best_val_0.6175_transformer_next_cluster_k80_16_200_0.2_8_0.3_256_3e-05_0.1_0.0.pt",
    #checkpoint_path="experiments_weights/best_val_0.7187_transformer_next_cluster_16_200_0.2_8_0.3_256_3e-05_0.1_0.0.pt",
    device="cuda",
    k_list=K_LIST,
    trigram_subset=True
)

print("Edges evaluated (transformer):", tot_tf)
print("Transformer Recall@k:", rec_tf)

C:\Users\atsumilab\AppData\Local\Temp\ipykernel_58232\761414267.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, map_location="cpu")


Edges evaluated (transformer): 6058
Transformer Recall@k: {1: 0.11175305381313964, 3: 0.17976229778804886, 5: 0.22713766919775502, 10: 0.31165401122482667, 20: 0.44189501485638827}


In [14]:
checkpoints=[]
checkpoints.append("experiments_weights/best_val_0.8236_transformer_next_cluster_k20_16_200_0.2_8_0.3_256_3e-05_0.1_0.0.pt")
checkpoints.append("experiments_weights/best_val_0.7187_transformer_next_cluster_k40_16_200_0.2_8_0.3_256_3e-05_0.1_0.0.pt")
checkpoints.append("experiments_weights/best_val_0.6175_transformer_next_cluster_k80_16_200_0.2_8_0.3_256_3e-05_0.1_0.0.pt")
checkpoints.append("experiments_weights/best_val_0.5535_transformer_next_cluster_k120_16_200_0.2_8_0.3_256_3e-05_0.1_0.0.pt")
checkpoints.append("experiments_weights/best_val_0.4683_transformer_next_cluster_k200_16_200_0.2_8_0.3_256_3e-05_0.1_0.0.pt")

#DATA_PATH = "processed_data/clusters/screen_clusters_k40.tsv"  # <-- change this

for i,kk in enumerate([20, 40, 80, 120, 200]):

    DATA_PATH = f"processed_data/clusters/screen_clusters_k{kk}.tsv"
    print(DATA_PATH)
    print("Checkpoint path:", i, checkpoints[i])


    transfomer_checkpoint_path = checkpoints[i]
    df= load_traces_tsv(DATA_PATH)
    traces_raw = build_traces(df, collapse_consecutive_duplicates=True)
    print("Num traces (len>=2):", len(traces_raw))

    traces, id2new, new2old, K = remap_clusters(traces_raw)
    print("K (unique clusters observed):", K)

    train_traces, test_traces, test_apps = app_split(traces, test_frac=0.2, seed=42)
    print("Train traces:", len(train_traces))
    print("Test traces :", len(test_traces))
    print("Test apps   :", len(test_apps))

    unigram_probs = train_unigram_next(train_traces, K, alpha=ALPHA_SMOOTH)
    bigram_probs  = train_bigram_counts(train_traces, K, alpha=ALPHA_SMOOTH)
    trigram_probs = train_trigram_counts(train_traces, K, alpha=ALPHA_SMOOTH)

    print("Trigram contexts:", len(trigram_probs))




    rec_u, tot_u = eval_recall_at_k_unigram(test_traces, unigram_probs, K_LIST, trigram_subset=True)
    rec_b, tot_b = eval_recall_at_k_bigram(test_traces, bigram_probs, K_LIST, trigram_subset=True)
    rec_t, tot_t = eval_recall_at_k_trigram(test_traces, trigram_probs, bigram_probs, unigram_probs,
                                            lambda_backoff=LAMBDA_BACKOFF, k_list=K_LIST)

    print("Edges evaluated (unigram):", tot_u)
    print("Edges evaluated (bigram) :", tot_b)
    print("Edges evaluated (trigram):", tot_t)


    rec_tf, tot_tf = eval_recall_at_k_transformer(
        test_traces,
        checkpoint_path=transfomer_checkpoint_path,
        device="cuda",
        k_list=K_LIST,
        trigram_subset=True
    )

    print("Edges evaluated (transformer):", tot_tf)
    print("Transformer Recall@k:", rec_tf)


    print("Unigram Recall@k:", rec_u)
    print("Bigram  Recall@k:", rec_b)
    print("Trigram Recall@k:", rec_t)

processed_data/clusters/screen_clusters_k20.tsv
Checkpoint path: 0 experiments_weights/best_val_0.8236_transformer_next_cluster_k20_16_200_0.2_8_0.3_256_3e-05_0.1_0.0.pt
Num traces (len>=2): 7831
K (unique clusters observed): 20
Train traces: 6271
Test traces : 1560
Test apps   : 1458
Trigram contexts: 378
Edges evaluated (unigram): 5774
Edges evaluated (bigram) : 5774
Edges evaluated (trigram): 5774


C:\Users\atsumilab\AppData\Local\Temp\ipykernel_58232\761414267.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, map_location="cpu")


Edges evaluated (transformer): 5774
Transformer Recall@k: {1: 0.3236924142708694, 3: 0.544163491513682, 5: 0.6527537235885001, 10: 0.8365084863179771, 20: 1.0}
Unigram Recall@k: {1: 0.1175961205403533, 3: 0.28368548666435744, 5: 0.41461724974021474, 10: 0.697783165916176, 20: 1.0}
Bigram  Recall@k: {1: 0.15881537928645653, 3: 0.3605819189470038, 5: 0.5083131278143401, 10: 0.7589192933841358, 20: 1.0}
Trigram Recall@k: {1: 0.2850710079667475, 3: 0.45704883962590925, 5: 0.5829580879806027, 10: 0.7914790439903013, 20: 1.0}
processed_data/clusters/screen_clusters_k40.tsv
Checkpoint path: 1 experiments_weights/best_val_0.7187_transformer_next_cluster_k40_16_200_0.2_8_0.3_256_3e-05_0.1_0.0.pt
Num traces (len>=2): 8005
K (unique clusters observed): 40
Train traces: 6397
Test traces : 1608
Test apps   : 1489
Trigram contexts: 1332
Edges evaluated (unigram): 6058
Edges evaluated (bigram) : 6058
Edges evaluated (trigram): 6058


C:\Users\atsumilab\AppData\Local\Temp\ipykernel_58232\761414267.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, map_location="cpu")


Edges evaluated (transformer): 6058
Transformer Recall@k: {1: 0.25982172334103665, 3: 0.4579068999669858, 5: 0.5643776824034334, 10: 0.7268075272367118, 20: 0.8951799273687686}
Unigram Recall@k: {1: 0.060911191812479365, 3: 0.15912842522284582, 5: 0.24694618686034994, 10: 0.4517992736876857, 20: 0.7256520303730604}
Bigram  Recall@k: {1: 0.12198745460548036, 3: 0.2692307692307692, 5: 0.38478045559590623, 10: 0.5747771541762958, 20: 0.8172664245625619}
Trigram Recall@k: {1: 0.21492241663915485, 3: 0.3626609442060086, 5: 0.463519313304721, 10: 0.6310663585341697, 20: 0.8367448002641136}
processed_data/clusters/screen_clusters_k80.tsv
Checkpoint path: 2 experiments_weights/best_val_0.6175_transformer_next_cluster_k80_16_200_0.2_8_0.3_256_3e-05_0.1_0.0.pt
Num traces (len>=2): 8096
K (unique clusters observed): 80
Train traces: 6468
Test traces : 1628
Test apps   : 1503
Trigram contexts: 4286
Edges evaluated (unigram): 6165
Edges evaluated (bigram) : 6165
Edges evaluated (trigram): 6165


...

Edges evaluated (transformer): 6165
Transformer Recall@k: {1: 0.24330900243309003, 3: 0.40227088402270883, 5: 0.49278183292781835, 10: 0.6283860502838605, 20: 0.775669099756691}
Unigram Recall@k: {1: 0.04395782643957826, 3: 0.11873479318734793, 5: 0.16901865369018654, 10: 0.2896999188969992, 20: 0.4742903487429035}
Bigram  Recall@k: {1: 0.09927007299270073, 3: 0.21751824817518248, 5: 0.30154095701540956, 10: 0.45012165450121655, 20: 0.6364963503649635}
Trigram Recall@k: {1: 0.15669099756690996, 3: 0.28629359286293593, 5: 0.36382806163828063, 10: 0.4932684509326845, 20: 0.6655312246553122}
processed_data/clusters/screen_clusters_k120.tsv
Checkpoint path: 3 experiments_weights/best_val_0.5535_transformer_next_cluster_k120_16_200_0.2_8_0.3_256_3e-05_0.1_0.0.pt
Num traces (len>=2): 8104
K (unique clusters observed): 120
Train traces: 6490
Test traces : 1614
Test apps   : 1505
Trigram contexts: 7200
Edges evaluated (unigram): 6292
Edges evaluated (bigram) : 6292
Edges evaluated (trigram): 6

C:\Users\atsumilab\AppData\Local\Temp\ipykernel_58232\761414267.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, map_location="cpu")


Edges evaluated (transformer): 6292
Transformer Recall@k: {1: 0.2029561347743166, 3: 0.35282898919262556, 5: 0.4359504132231405, 10: 0.5588048315321043, 20: 0.6953273998728544}
Unigram Recall@k: {1: 0.033693579148124604, 3: 0.08598219961856325, 5: 0.12730451366815004, 10: 0.21439923712650985, 20: 0.3631595677050222}
Bigram  Recall@k: {1: 0.0875715193897012, 3: 0.19326128417037508, 5: 0.26478067387158294, 10: 0.38922441195168467, 20: 0.5416401780038144}
Trigram Recall@k: {1: 0.1217418944691672, 3: 0.2358550540368722, 5: 0.3094405594405594, 10: 0.41926255562619197, 20: 0.5684996821360457}
processed_data/clusters/screen_clusters_k200.tsv
Checkpoint path: 4 experiments_weights/best_val_0.4683_transformer_next_cluster_k200_16_200_0.2_8_0.3_256_3e-05_0.1_0.0.pt
Num traces (len>=2): 8158
K (unique clusters observed): 200
Train traces: 6517
Test traces : 1641
Test apps   : 1515
Trigram contexts: 11626
Edges evaluated (unigram): 6999
Edges evaluated (bigram) : 6999
Edges evaluated (trigram): 69

C:\Users\atsumilab\AppData\Local\Temp\ipykernel_58232\761414267.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, map_location="cpu")


Edges evaluated (transformer): 6999
Transformer Recall@k: {1: 0.17831118731247322, 3: 0.30947278182597515, 5: 0.3810544363480497, 10: 0.49564223460494355, 20: 0.6106586655236462}
Unigram Recall@k: {1: 0.017145306472353194, 3: 0.04929275610801543, 5: 0.0770110015716531, 10: 0.13844834976425205, 20: 0.24932133161880268}
Bigram  Recall@k: {1: 0.07343906272324618, 3: 0.15802257465352193, 5: 0.22031718816973853, 10: 0.31561651664523505, 20: 0.44363480497213886}
Trigram Recall@k: {1: 0.0958708386912416, 3: 0.1881697385340763, 5: 0.24974996428061152, 10: 0.3416202314616374, 20: 0.46206600942991854}


In [10]:

print("Unigram Recall@k:", rec_u)
print("Bigram  Recall@k:", rec_b)
print("Trigram Recall@k:", rec_t)

Unigram Recall@k: {1: 0.060911191812479365, 3: 0.15912842522284582, 5: 0.24694618686034994, 10: 0.4517992736876857, 20: 0.7256520303730604}
Bigram  Recall@k: {1: 0.12198745460548036, 3: 0.2692307692307692, 5: 0.38478045559590623, 10: 0.5747771541762958, 20: 0.8172664245625619}
Trigram Recall@k: {1: 0.21492241663915485, 3: 0.3626609442060086, 5: 0.463519313304721, 10: 0.6310663585341697, 20: 0.8367448002641136}


In [6]:
for k,val in rec_tf.items():
    print(f"Transformer Recall@{k}: {val:.4f}")

Transformer Recall@1: 0.2598
Transformer Recall@3: 0.4579
Transformer Recall@5: 0.5644
Transformer Recall@10: 0.7268
Transformer Recall@20: 0.8952


--------